# 10-714 作业 2

本次作业将基于 Needle 框架实现一个神经网络库。提醒：**请务必在 Google Drive 中保存一份副本。**

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw2.git
%cd /content/drive/MyDrive/10714/hw2

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

### 设置一些变量

In [ ]:
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW2_NAME = "hw2"

## 问题 0

本次作业基于作业 1 的内容。

**首先，请将作业 1 中的 `python/needle/autograd.py` 和 `python/needle/ops/ops_mathematic.py` 复制到作业 2 目录中。**

***注意***：Tensor 的默认数据类型是 `float32`。如果要更改数据类型，可以在 `Tensor` 构造函数中设置 `dtype` 参数。例如，`Tensor([1, 2, 3], dtype='float64')` 会创建一个数据类型为 `float64` 的张量。
在本次作业中，**请确保创建的所有张量都使用 `float32` 数据类型，以避免自动评分器出现问题。**

In [ ]:
import sys
sys.path.append('./python')
sys.path.append('./apps')

## 问题 1

本题将实现几种不同的权重初始化方法。相关代码位于 `python/needle/init/init_initializers.py`，其中包含使用随机值和常量初始化 Needle Tensor 的多个函数。请参照已有初始化函数的写法（例如，在下面的函数中调用已在 `python/needle/init/init_basic.py` 实现的 `init.rand` 或 `init.randn`），实现以下常见初始化方法。所有函数都应返回形状为 `fan_in` × `fan_out` 的二维张量（如需扩展到其他尺寸，可以通过重塑等方式实现）。

### Xavier 均匀初始化
`xavier_uniform(fan_in, fan_out, gain=1.0, **kwargs)`

按照论文 [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf) 中的方法，使用均匀分布填充输入 Tensor。生成的 Tensor 中的值从以下分布采样：

$$a = \text{gain} \times \sqrt{\frac{6}{\text{fan\_in} + \text{fan\_out}}}$$

即服从 $\mathcal{U}(-a, a)$。

将剩余的 `**kwargs` 参数传递给对应的 `init` 随机初始化函数。

##### 参数
- `fan_in`：输入维度
- `fan_out`：输出维度
- `gain`：可选的缩放系数
___

### Xavier 正态初始化
`xavier_normal(fan_in, fan_out, gain=1.0, **kwargs)`

按照上述论文中的方法，使用正态分布填充输入 Tensor。生成的 Tensor 中的值服从 $\mathcal{N}(0, \text{std}^2)$，其中：

$$\mathrm{std} = \mathrm{gain} \times \sqrt{\frac{2}{\mathrm{fan}_{in} + \mathrm{fan}_{out}}}$$

##### 参数
- `fan_in`：输入维度
- `fan_out`：输出维度
- `gain`：可选的缩放系数
___

### Kaiming 均匀初始化
`kaiming_uniform(fan_in, fan_out, nonlinearity="relu", **kwargs)`

按照论文 [Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification](https://arxiv.org/pdf/1502.01852.pdf) 中的方法，使用均匀分布填充输入 Tensor。生成的 Tensor 中的值服从 $\mathcal{U}(-\text{bound}, \text{bound})$，其中：

$$\mathrm{bound} = \mathrm{gain} \times \sqrt{\frac{3}{\mathrm{fan}_{in}}}$$

ReLU 推荐使用 $\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in`：输入维度
- `fan_out`：输出维度
- `nonlinearity`：非线性函数的名称
___

### Kaiming 正态初始化
`kaiming_normal(fan_in, fan_out, nonlinearity="relu", **kwargs)`

按照上述论文中的方法，使用正态分布填充输入 Tensor。生成的 Tensor 中的值服从 $\mathcal{N}(0, \text{std}^2)$，其中：

$$\mathrm{std} = \frac{\mathrm{gain}}{\sqrt{\mathrm{fan}_{in}}}$$

ReLU 推荐使用 $\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in`：输入维度
- `fan_out`：输出维度
- `nonlinearity`：非线性函数的名称

In [ ]:
!python3 -m pytest -v -k "test_init"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "init"

## 问题 2

本题需要在 `python/needle/nn/nn_basic.py` 中实现更多模块。具体来说，请在构造函数中初始化各模块所需的变量，并完成 `forward` 方法。**注意：**请使用刚刚实现的 `init` 函数初始化参数，并记得传入 `dtype` 参数。
___

### Linear（线性层）
`needle.nn.Linear(in_features, out_features, bias=True, device=None, dtype="float32")`

对输入数据进行线性变换：$y = xA^T + b$。输入形状为 $(N, H_{\text{in}})$，其中 $H_{\text{in}}=\text{in\_features}$；输出形状为 $(N, H_{\text{out}})$，其中 $H_{\text{out}}=\text{out\_features}$。

**请注意，必须显式地将 bias 广播到正确的形状；Needle 不支持隐式广播。**

**注意：包括本层在内，所有层都应先初始化 weight Tensor，再初始化 bias Tensor；并且所有 Parameter 都只能使用 `init` 中的函数进行初始化。**这是因为 mugrade 中的测试答案假定权重先于偏置初始化。虽然先初始化偏置在算法上也成立，但模型解答和预期测试结果使用了“先权重、后偏置”的顺序。要通过 mugrade 自动测试，必须遵循这个顺序。如果本作业其他部分中某一层或参数的初始化顺序不明确，请在 Ed 上提问。

##### 参数
- `in_features`：每个输入样本的维度
- `out_features`：每个输出样本的维度
- `bias`：若设为 `False`，该层不会学习加性偏置

##### 变量
- `weight`：形状为 (`in_features`, `out_features`) 的可学习权重。请使用 `fan_in = in_features` 的 Kaiming 均匀初始化。
- `bias`：形状为 (`out_features`,) 的可学习偏置。请使用 `fan_in = out_features` 的 Kaiming 均匀初始化。**注意这里选择的 fan_in 与 weight 不同，这是由它们各自的尺寸决定的。**

**注意：**请将所有必要变量（例如 `weight`、`bias`）封装在 `Parameter` 类中，这样优化器才能找到它们。可以阅读 `python/needle/nn/nn_basic.py` 中的 `class Parameter` 和 `unpack_params` 函数了解更多。

In [ ]:
!python3 -m pytest -v -k "test_nn_linear"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_linear"

### ReLU
`needle.nn.ReLU()`

逐元素应用修正线性单元函数：
$ReLU(x) = max(0, x)$。

如果你之前曾用 ReLU 本身来表达它的反向传播过程，需要注意这种做法在数值上不稳定，后续可能会引发问题。可以将 ReLU 的导数写为 $I\{x>0\}$。本作业约定在 $x=0$ 处的导数为 0。（这是一个**次可微**函数。）

___

In [ ]:
!python3 -m pytest -v -k "test_nn_relu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_relu"

### Sequential（顺序容器）
`needle.nn.Sequential(*modules)`

按照构造函数中传入的顺序，将输入依次传递给各个模块，并返回最后一个模块的输出。模块应保存在 `.module` 属性中；**不要重新定义 `__getitem__` 等魔术方法**，因为这样可能与测试不兼容。

##### 参数
- `*modules`：任意数量的 `needle.nn.Module` 类型模块

___

In [ ]:
!python3 -m pytest -v -k "test_nn_sequential"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_sequential"

### LogSumExp

`needle.ops.LogSumExp(axes)`

通过减去最大元素，实现数值稳定的 log-sum-exp 函数。你需要在 `python/needle/ops/ops_logarithmic.py` 文件中实现本操作以及下一个操作。

\begin{equation}
\text{LogSumExp}(z) = \log (\sum_{i} \exp (z_i - \max{z})) + \max{z}
\end{equation}

#### 参数
- `axes`：需要进行求和并取最大值的轴元组，使用方式与 `needle.ops.Summation()` 相同。

##### 为什么使用这种形式？数值稳定性处理

- **朴素定义：**最直接的 LogSumExp 定义为

  $$
  \log \left(\sum_i \exp(z_i)\right)
  $$

- **问题：**这种直接计算容易出现数值不稳定。
  - 如果某个 $z_i$ 很大（例如 $1000$），那么 $\exp(z_i)$ 可能溢出为 $\infty$。
  - 如果某个 $z_i$ 很小（例如 $-1000$），那么 $\exp(z_i)$ 可能下溢为 $0$。
  这两种情况都会使浮点计算结果失真。

- **解决方法：**提取最大元素 $M = \max(z)$：

  $$
  \log \left(\sum_i \exp(z_i)\right)
  = \log \left(\exp(M)\sum_i \exp(z_i - M)\right)
  = M + \log \left(\sum_i \exp(z_i - M)\right).
  $$

  这样所有指数项都不大于 $\exp(0) = 1$，因此可以避免上溢。

- **下溢会怎样？**如果 $z_i - M$ 非常小（例如 $-1000$），指数仍可能下溢，因为浮点数中的 $\exp(-1000) \approx 0$。但这不会造成问题：与最大值相比，这些项已经很小，对总和没有实质影响。

下面的博客文章也很有参考价值：https://indii.org/blog/gradients-of-softmax-and-logsumexp/
___

In [ ]:
!python3 -m pytest -v -k "test_op_logsumexp"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsumexp"

### LogSoftmax

`needle.ops.LogSoftmax(axes)`

通过减去最大元素，实现数值稳定的 logsoftmax 函数。本题可以假设输入 NDArray 是二维的，并且沿 `axis=1` 计算 softmax。

\begin{equation}
\text{LogSoftmax}(z) = \log \left(\frac{\exp(z_i - \max z)}{\sum_{i}\exp(z_i - \max z)}\right) = z - \text{LogSumExp}(z)
\end{equation}
___

In [ ]:
!python3 -m pytest -v -k "test_op_logsoftmax"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsoftmax"

### SoftmaxLoss

`python/needle/nn/nn_basic.py` 中的 `needle.nn.SoftmaxLoss()`

按照下方定义（与作业 1 中的实现相同）计算 softmax 损失。输入为 logits Tensor 和真实标签 Tensor；真实标签以类别编号列表表示，**不是 one-hot 编码**。

现在可以使用 `init.one_hot` 函数，不必自行实现 one-hot 编码。注意：本题需要使用刚刚实现的数值稳定 LogSumExp 算子。

\begin{equation}
\ell_\text{softmax}(z,y) = \log \sum_{i=1}^k \exp z_i - z_y
\end{equation}

___

In [ ]:
!python3 -m pytest -v -k "test_nn_softmax_loss"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_softmax_loss"

### LayerNorm1d
`needle.nn.LayerNorm1d(dim, eps=1e-5, device=None, dtype="float32")`

按照论文 [Layer Normalization](https://arxiv.org/abs/1607.06450) 中的描述，对一批输入进行层归一化。

\begin{equation}
y = w \circ \frac{x_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

其中，$\textbf{E}[x]$ 表示输入的经验均值，$\textbf{Var}[x]$ 表示输入的经验方差（这里使用有偏方差估计，即除以 $N$ 而不是 $N-1$），$w$ 和 $b$ 分别表示可学习的缩放权重和偏置。可以假设本层输入是二维张量，第一维为批次，第二维为特征。应用权重和偏置前，可能需要先对它们进行广播。

##### 参数
- `dim`：通道数
- `eps`：加到分母中的数值，用于保证数值稳定

##### 变量
- `weight`：大小为 `dim` 的可学习权重，初始值为 1。
- `bias`：形状为 `dim` 的可学习偏置，初始值为 0。
___

In [ ]:
!python3 -m pytest -v -k "test_nn_layernorm"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_layernorm"

### Flatten（展平）
`needle.nn.Flatten()`

输入形状为 `(B,X_0,X_1,...)` 的张量，将除批次维度以外的所有维度展平，输出形状为 `(B, X_0 * X_1 * ...)`。

In [ ]:
!python3 -m pytest -v -k "test_nn_flatten"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_flatten"

### BatchNorm1d
`needle.nn.BatchNorm1d(dim, eps=1e-5, momentum=0.1, device=None, dtype="float32")`

按照论文 [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167) 中的描述，对一个小批次的输入进行批归一化。

\begin{equation}
y = w \circ \frac{z_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

这里的均值和方差是在**批次维度**上计算的。该函数还会为每一层的所有特征计算均值和方差的移动平均 $\hat{\mu}, \hat{\sigma}^2$；测试时使用这些量进行归一化：

\begin{equation}
y = \frac{(x - \hat{mu})}{((\hat{\sigma}^2_{i+1})_j+\epsilon)^{1/2}}
\end{equation}

测试时，BatchNorm 使用运行中的均值和方差估计值，而不是当前批次的统计量。调用 BatchNorm 层所属模型的 `model.eval()` 后，该层的 `training` 标志会变为 `false`。

可以使用下式计算运行中的估计值：$$\hat{x_{new}} = (1 - m) \hat{x_{old}} + mx_{observed},$$其中 $m$ 是 momentum。

##### 参数
- `dim`：输入维度
- `eps`：加到分母中的数值，用于保证数值稳定
- `momentum`：计算运行均值和运行方差时使用的系数

##### 变量
- `weight`：大小为 `dim` 的可学习权重，初始值为 1。
- `bias`：大小为 `dim` 的可学习偏置，初始值为 0。
- `running_mean`：评估时使用的运行均值，初始值为 0。
- `running_var`：评估时使用的运行方差（有偏方差），初始值为 1。

注意：为简化实现，Needle 对有偏方差使用运行平均；PyTorch 在评估时使用无偏估计。
___

In [ ]:
!python3 -m pytest -v -k "test_nn_batchnorm"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_batchnorm"

### Dropout
`needle.nn.Dropout(p = 0.5)`

训练时，从伯努利分布采样，并以概率 `p` 将输入张量中的部分元素随机置零。这是一种有效的正则化方法，可以防止神经元之间的协同适应，相关方法见论文 [Improving neural networks by preventing co-adaptation of feature detectors](https://arxiv.org/abs/1207.0580)。评估时，该模块只执行恒等映射。

\begin{equation}
\hat{z}_{i+1} = \sigma_i (W_i^T z_i + b_i) \\
(z_{i+1})_j =
    \begin{cases}
    (\hat{z}_{i+1})_j /(1-p) & \text{概率为 } 1-p \\
    0 & \text{概率为 } p \\
    \end{cases}
\end{equation}

除以 (1-p) 可以保持激活值的期望不变，因为

$$
\mathbb{E}\big[\operatorname{Dropout}((\hat{z}_{i+1})_j)\big]
= (1-p)\,\frac{(\hat{z}_{i+1})_j}{1-p} + p \cdot 0
= (\hat{z}_{i+1})_j
$$

**重要：**如果 Dropout 模块的 `training=False`，就不应丢弃任何激活值。也就是说，Dropout 只在训练阶段生效，评估阶段不生效。`training` 是 `nn.Module` 中的一个标志。

##### 参数
- `p`：一个元素被置零的概率

实现 Dropout 时，`python/needle/init/init_basic.py` 中的工具函数可能会有帮助。
___

In [ ]:
!python3 -m pytest -v -k "test_nn_dropout"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_dropout"

### Residual（残差连接）
`needle.nn.Residual(fn: Module)`

给定模块 $\mathcal{F}$ 和输入 Tensor $x$，应用残差（跳跃）连接并返回 $\mathcal{F}(x) + x$。

##### 参数
- `fn`：`needle.nn.Module` 类型的模块

In [ ]:
!python3 -m pytest -v -k "test_nn_residual"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_residual"

## 问题 3

请在 `python/needle/optim.py` 中实现以下优化器的 `step` 函数。
确保优化器**不会原地修改** Tensor 的梯度。

我们提供了一些测试，以确保实现不会消耗过多内存。如果没有在合适的位置使用 `.data` 或 `.detach()`，计算图可能会不断增长，从而造成内存问题（不仅优化器中可能发生，前面实现的模块中也可能发生）。你可以自行决定是否运行名称中包含 `memory_check` 的测试。

___

### SGD
`needle.optim.SGD(params, lr=0.01, momentum=0.0, weight_decay=0.0)`

实现随机梯度下降（可选动量，以下用 $\beta$ 表示）。

\begin{equation}
\begin{split}
    u_{t+1} &= \beta u_t + (1-\beta) \nabla_\theta f(\theta_t) \\
    \theta_{t+1} &= \theta_t - \alpha u_{t+1}
\end{split}
\end{equation}

##### 参数
- `params`：待优化的 `needle.nn.Parameter` 参数迭代器
- `lr`（*float*）：学习率
- `momentum`（*float*）：动量系数
- `weight_decay`（*float*）：权重衰减（L2 惩罚）

本次作业可以跳过 `clip_grad_norm` 的实现。
___

In [ ]:
!python3 -m pytest -v -k "test_optim_sgd"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_sgd"

### Adam
`needle.optim.Adam(params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0)`

实现论文 [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) 提出的 Adam 算法。

\begin{equation}
\begin{split}
u_{t+1} &= \beta_1 u_t + (1-\beta_1) \nabla_\theta f(\theta_t) \\
v_{t+1} &= \beta_2 v_t + (1-\beta_2) (\nabla_\theta f(\theta_t))^2 \\
\hat{u}_{t+1} &= u_{t+1} / (1 - \beta_1^t) \quad \text{（偏差修正）} \\
\hat{v}_{t+1} &= v_{t+1} / (1 - \beta_2^t) \quad \text{（偏差修正）}\\
\theta_{t+1} &= \theta_t - \alpha \hat{u_{t+1}}/(\hat{v}_{t+1}^{1/2}+\epsilon)
\end{split}
\end{equation}

**重要：**请注意是否需要应用偏差修正。

##### 参数
- `params`：待优化的 `needle.nn.Parameter` 参数迭代器
- `lr`（*float*）：学习率
- `beta1`（*float*）：计算梯度运行平均值时使用的系数
- `beta2`（*float*）：计算梯度平方运行平均值时使用的系数
- `eps`（*float*）：加到分母中的数值，用于改善数值稳定性
- `weight_decay`（*float*）：权重衰减（L2 惩罚）

**提示：**为避免内存问题，请尝试理解 `.data` 和 `.detach()` 的用法。

In [ ]:
!python3 -m pytest -v -k "test_optim_adam"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_adam"

## 问题 4

本题需要实现两个数据处理基础组件：`needle.data.DataLoader` 和 `needle.data.Dataset`。`Dataset` 保存样本及其对应标签；`DataLoader` 将 `Dataset` 包装为可迭代对象，方便访问样本。

本题涉及 `python/needle/data` 目录。

### 数据变换

首先实现一些图像处理中常用的变换。目前只实现水平翻转和随机裁剪。请补全 `needle/data/data_transforms.py` 中的对应函数。
___

#### RandomFlipHorizontal（随机水平翻转）
`needle.data.RandomFlipHorizontal(p = 0.5)`

以概率 `p` 水平翻转图像。

##### 参数
- `p`（*float*）：翻转输入图像的概率
___

#### RandomCrop（随机裁剪）
`needle.data.RandomCrop(padding=3)`

在图像四周添加填充，然后从随机位置裁剪回原始尺寸。返回图像的尺寸与原图相同。

##### 参数
- `padding`（*int*）：图像每条边上的填充宽度

In [ ]:
!python3 -m pytest -v -k "flip_horizontal"
!python3 -m pytest -v -k "random_crop"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "flip_horizontal"
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "random_crop"

Dataset 负责定义数据的内容（例如图像和标签对）以及如何获取单个样本。DataLoader 负责定义如何将数据送入训练流程（例如分批、打乱和按轮次迭代）。

### Dataset

`Dataset` 的每个**子类**都必须实现三个函数：`__init__`、`__len__` 和 `__getitem__`。
* `__init__` 函数初始化图像、标签和变换。
* `__len__` 函数返回样本数量。
* `__getitem__` 函数根据给定索引 `idx` 获取一个样本，**并在适用时对图像调用变换函数**；随后将图像和标签转换为 NumPy 数组（数据会在其他位置转换为 Tensor）。`__getitem__` 和 `__next__` 的输出都应为 NDArray。访问数组时请遵循 `(数据点数量, 特征维度 1, 特征维度 2, ...)` 的形状约定。

请在 `needle/data/datasets/mnist_dataset.py` 中的 `MNISTDataset` 类里完成这些函数。其 `__init__` 函数可以复用你在上一份作业中实现的 `parse_mnist`。

### MNISTDataset（Dataset 的子类）
`needle.data.MNISTDataset(image_filename, label_filename, transforms)`

##### 参数
- `image_filename`：图像文件路径
- `label_filename`：标签文件路径
- `transforms`：可选的变换列表，用于处理数据

In [ ]:
!python3 -m pytest -v -k "test_mnist_dataset"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mnist_dataset"

### Dataloader

`needle.data.Dataloader(dataset: Dataset, batch_size: Optional[int] = 1, shuffle: bool = False)`

在 `needle/data/data_basic.py` 中，Dataloader 类提供了一种接口：基于 `Dataset` 对象组装适用于 SGD 训练的小批次样本。为了实现常见的 Dataloader 接口（可以遍历数据集中的所有小批次），你需要实现类中的 `__iter__()` 和 `__next__()`：
* 每个新训练轮次开始时（即开始遍历 dataloader 时）会调用 `__iter__()`。
* 每获取一个批次会调用一次 `__next__()`，直到本轮结束。

请注意，后续多次调用 `next` 时都需要返回下一个批次，因此 `next` 不是纯函数。

##### 用途

组合一个数据集和一个采样器，并提供对给定数据集的可迭代访问。

##### 参数
- `dataset`：`needle.data.Dataset` 类型的数据集
- `batch_size`：每次提供的数据批次大小
- `shuffle`：`bool` 类型；若设为 `True`，则在**每个训练轮次开始时**重新打乱数据，默认为 `False`
___

In [ ]:
!python3 -m pytest -v -k "test_dataloader"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "dataloader"

## 问题 5

现在你已经实现神经网络库所需的全部组件，可以开始构建并训练 MLP ResNet。本题需要修改 `apps/mlp_resnet.py`。首先，按照以下说明补全 `ResidualBlock` 和 `MLPResNet` 函数：

### ResidualBlock
`ResidualBlock(dim, hidden_dim, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现如下残差块：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/residualblock.png?raw=true" alt="Residual Block"/>
</p>

**注意**：如果图片无法显示，请查看 `figures` 目录中的图片。

其中第一个线性层的 `in_features=dim`、`out_features=hidden_dim`，最后一个线性层的 `out_features=dim`。返回类型为 `nn.Module` 的残差块。

##### 参数
- `dim`（*int*）：输入维度
- `hidden_dim`（*int*）：隐藏层维度
- `norm`（*nn.Module*）：归一化方法
- `drop_prob`（*float*）：Dropout 概率

___

### MLPResNet
`MLPResNet(dim, hidden_dim=100, num_blocks=3, num_classes=10, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现如下 MLP ResNet：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/mlp_resnet.png?raw=true" alt="MLP Resnet"/>
</p>

其中第一个线性层的 `in_features=dim`、`out_features=hidden_dim`；每个 ResidualBlock 的 `dim=hidden_dim`、`hidden_dim=hidden_dim//2`。返回类型为 `nn.Module` 的网络。

##### 参数
- `dim`（*int*）：输入维度
- `hidden_dim`（*int*）：隐藏层维度
- `num_blocks`（*int*）：ResidualBlock 的数量
- `num_classes`（*int*）：类别数
- `norm`（*nn.Module*）：归一化方法
- `drop_prob`（*float*）：Dropout 概率（0.1）

**注意**：模块的初始化顺序应与 ResNet 中的执行顺序一致。
___

网络结构正确后，使用新实现的神经网络库训练模型。具体来说，请实现 `epoch` 和 `train_mnist` 函数。

### Epoch

`epoch(dataloader, model, opt=None)`

执行一轮训练或评估，遍历整个训练数据集一次（与之前作业中的 `nn_epoch` 类似）。返回平均错误率（*float*）和所有样本的平均损失（*float*）。如果提供了 `opt`，则在函数开始时将模型设为训练模式；如果没有提供优化器（即 `None`），则设为评估模式。切换模式时请调用 `.train()` 和 `.eval()`，不要直接修改 `training` 属性。

##### 参数
- `dataloader`（`needle.data.DataLoader`）：返回训练数据样本的 dataloader
- `model`（`needle.nn.Module`）：神经网络
- `opt`（`needle.optim.Optimizer`）：优化器实例，或 `None`

___

### Train Mnist

`train_mnist(batch_size=100, epochs=10, optimizer=ndl.optim.Adam, lr=0.001, weight_decay=0.001, hidden_dim=100, data_dir="data")`

初始化 MNIST 训练 dataloader（`shuffle` 设为 `True`）和测试 dataloader，并使用给定的优化器（如果 `opt` 不为 `None`）与 softmax 损失训练 MLPResNet 指定轮数。返回最后一轮训练得到的训练错误率、训练损失、测试错误率和测试损失组成的元组。未指定参数时使用默认值。

##### 参数
- `batch_size`（*int*）：训练和测试 dataloader 使用的批次大小
- `epochs`（*int*）：训练轮数
- `optimizer`（`needle.optim.Optimizer` 类型）：使用的优化器类型
- `lr`（*float*）：学习率
- `weight_decay`（*float*）：权重衰减
- `hidden_dim`（*int*）：MLPResNet 的隐藏层维度
- `data_dir`（*str*）：存放 MNIST 图像和标签文件的目录

In [ ]:
!python3 -m pytest -v -k "test_mlp"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mlp_resnet"

我们鼓励你尝试运行 `mlp_resnet.py` 训练脚本进行实验。
你可以研究在线性层中使用不同初始化方法的效果、提高 dropout 概率，
或者添加数据变换（通过 Dataset 的 `transforms=` 关键字参数传入变换列表），
例如随机裁剪。